In [1]:
import numpy as numpy
import sklearn as sk
import pandas as pd

In [4]:
df = pd.read_csv('cattle_data_train.csv')
print(len(df))


210000


Data Cleaning

In [8]:
labels = df["Milk_Yield_L"]

data = df.drop(columns=["Cattle_ID","Date","Farm_ID","Milk_Yield_L","Feeding_Frequency","Housing_Score"])

# data = data.dropna() # Just drop all NaNs for now
print(data.head())

categorical_cols = [c for c in data.columns if data[c].dtype == 'object']
numeric_cols = [c for c in data.columns if c not in categorical_cols]

print(categorical_cols)
print(numeric_cols)
numeric_cols.append("Milk_Yield_L")
print(df[numeric_cols].corr()["Milk_Yield_L"])

numeric_transformer = sk.pipeline.Pipeline(steps=[
    ('imputer', sk.impute.SimpleImputer(strategy='median')),
    ('scaler', sk.preprocessing.StandardScaler())
])


categorical_transformer = sk.pipeline.Pipeline(steps=[
    ('imputer', sk.impute.SimpleImputer(strategy='most_frequent')),
    ('onehot', sk.preprocessing.OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = sk.compose.ColumnTransformer(transformers=[("numeric",numeric_transformer,numeric_cols),
("categorical",categorical_transformer,categorical_cols)])


      Breed   Climate_Zone Management_System  Age_Months  Weight_kg  Parity  \
0  Holstein       Tropical         Intensive         114      544.8       4   
1  Holstein           Arid             Mixed         136      298.9       4   
2  Holstein       Tropical    Semi_Intensive          64      336.6       4   
3    Jersey  Mediterranean         Intensive          58      370.5       1   
4  Guernsey    Subtropical         Intensive          84      641.5       6   

  Lactation_Stage  Days_in_Milk      Feed_Type  Feed_Quantity_kg  ...  \
0             Mid            62   Concentrates         16.363455  ...   
1             Mid           213  Crop_Residues               NaN  ...   
2            Late            16            Hay          7.198607  ...   
3           Early           339  Crop_Residues         18.694344  ...   
4           Early           125     Mixed_Feed         14.779198  ...   

   BQ_Vaccine  Anthrax_Vaccine  IBR_Vaccine  BVD_Vaccine  Rabies_Vaccine  \
0         

In [ ]:
pca = sk.decomposition.PCA(n_components=.9)

svm = sk.svm.LinearSVR()
pipeline = sk.pipeline.Pipeline(steps=[("pre",preprocessor),("pca",pca),("svm",svm)])
#"pca__n_components":[x/100 for x in range(80,100,5)],
params = {"svm__C":[.5,1,2,5,10,20,50]}

grid = sk.model_selection.GridSearchCV(pipeline,param_grid=params,scoring="neg_root_mean_squared_error")
grid.fit(data,labels)
print(grid.best_score_)
print(grid.best_params_)
# score = sk.model_selection.cross_val_score(pipeline,data,labels,scoring="neg_root_mean_squared_error")
# print(score)

c:\Users\Vincent Xia\miniconda3\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\Vincent Xia\miniconda3\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\Vincent Xia\miniconda3\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\Vincent Xia\miniconda3\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\Vincent Xia\miniconda3\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\Vincent Xia\miniconda3\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinea

-4.313240636368383
{'svm__C': 2}


c:\Users\Vincent Xia\miniconda3\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Build model with best params

In [6]:
svm = sk.svm.LinearSVR(C=2)
pipeline = sk.pipeline.Pipeline(steps=[("pre",preprocessor),("pca",pca),("svm",svm)])
model = pipeline.fit(data,labels)
score = sk.model_selection.cross_val_score(pipeline,data,labels,scoring="neg_root_mean_squared_error")
print(score)

c:\Users\Vincent Xia\miniconda3\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\Vincent Xia\miniconda3\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\Vincent Xia\miniconda3\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\Vincent Xia\miniconda3\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\Vincent Xia\miniconda3\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\Vincent Xia\miniconda3\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinea

[-4.29960938 -4.31175095 -4.3271947  -4.32744859 -4.30043266]


In [17]:
test = pd.read_csv('cattle_data_test.csv')
inputs = test.drop(columns=["Cattle_ID","Date","Farm_ID"])
predictions = model.predict(inputs)
test["Milk_Yield_L"]=predictions
test[["Cattle_ID","Milk_Yield_L"]].to_csv("./results.csv",index=False)

In [18]:
import pickle
pickle.dump(model,open("./final_model.sav","wb"))